In [1]:
import pathlib
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers, callbacks

# ── Constants ────────────────────────────────────────────────
RANDOM_SEED  = 42
SEQUENCE_LEN = 60
NUM_FEATURES = 126
BATCH_SIZE   = 32
EPOCHS       = 100

DATA_DIR  = pathlib.Path("../data/hand_v2")
MODEL_DIR = pathlib.Path("../saved_models/v2")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"TensorFlow : {tf.__version__}")

TensorFlow : 2.21.0


In [2]:
X_train = np.load(DATA_DIR / "X_train_aug.npy")
y_train = np.load(DATA_DIR / "y_train_aug.npy")

X_val   = np.load(DATA_DIR / "X_val_hand.npy")
y_val   = np.load(DATA_DIR / "y_val_hand.npy")

X_test  = np.load(DATA_DIR / "X_test_hand.npy")
y_test  = np.load(DATA_DIR / "y_test_hand.npy")

NUM_CLASSES = y_train.shape[1]

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Classes : {NUM_CLASSES}")

X_train : (10404, 60, 126)
X_val   : (247, 60, 126)
X_test  : (496, 60, 126)
Classes : 204


In [3]:
def topk_accuracy(model, X, y, k):
    preds = model.predict(X, verbose=0)
    true_labels = np.argmax(y, axis=1)
    topk_preds = np.argsort(preds, axis=1)[:, -k:]
    correct = sum(t in p for t, p in zip(true_labels, topk_preds))
    return correct / len(true_labels)


def eval_model(model, X, y, name):
    _, top1 = model.evaluate(X, y, verbose=0)
    top3 = topk_accuracy(model, X, y, k=3)
    top5 = topk_accuracy(model, X, y, k=5)
    print(f"  {name:<25}  Top-1: {top1*100:.2f}%   Top-3: {top3*100:.2f}%   Top-5: {top5*100:.2f}%")
    return top1, top3, top5

In [4]:
def build_gru(sequence_len, num_features, num_classes):
    """
    Lightweight unidirectional GRU companion.
    Input (60, 126)
    → GRU(128, return_sequences=True) + BatchNorm + Dropout(0.3)
    → GRU(64) + BatchNorm + Dropout(0.3)
    → Dense(128, relu) + Dropout(0.3)
    → Dense(num_classes, softmax)
    """
    inp = keras.Input(shape=(sequence_len, num_features))
    x = layers.GRU(128, return_sequences=True)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.GRU(64)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs=inp, outputs=out, name="gru_v2")


gru_model = build_gru(SEQUENCE_LEN, NUM_FEATURES, NUM_CLASSES)
gru_model.summary()

Model: "gru_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 60, 126)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 60, 128)        │        98,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 60, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 204)            │        26,316 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 170,956 (667.80 KB)

 Trainable params: 170,572 (666.30 KB)

 Non-trainable params: 384 (1.50 KB)

In [5]:
gru_checkpoint = str(MODEL_DIR / "gru_hand_v2_best.keras")

gru_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

gru_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True),
        callbacks.ModelCheckpoint(gru_checkpoint, monitor="val_accuracy", save_best_only=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6),
    ],
    verbose=1,
    )
    
gru_model = keras.models.load_model(gru_checkpoint)
print(f"GRU model saved: {gru_checkpoint}")

Epoch 1/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.0655 - loss: 4.8405 - val_accuracy: 0.0931 - val_loss: 4.4592 - learning_rate: 0.0010
Epoch 2/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.1392 - loss: 3.9795 - val_accuracy: 0.2186 - val_loss: 3.5793 - learning_rate: 0.0010
Epoch 3/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.2238 - loss: 3.1998 - val_accuracy: 0.3320 - val_loss: 3.0709 - learning_rate: 0.0010
Epoch 4/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.3164 - loss: 2.6024 - val_accuracy: 0.3806 - val_loss: 2.7550 - learning_rate: 0.0010
Epoch 5/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.3955 - loss: 2.1737 - val_accuracy: 0.3927 - val_loss: 2.6249 - learning_rate: 0.0010
Epoch 6/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.4694 - loss: 1.8154 - val_accuracy: 0.4332 - val_loss: 2.5101 - learning_rate: 0.0010
Epoch 7/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.5358 - l

In [6]:
def build_cnn(sequence_len, num_features, num_classes):
    """
    1D-CNN companion model.
    Input (60, 126)
    → Conv1D(64, 3) + BatchNorm + MaxPool(2) + Dropout(0.2)
    → Conv1D(128, 3) + BatchNorm + MaxPool(2) + Dropout(0.2)
    → GlobalAveragePooling1D
    → Dense(128, relu) + Dropout(0.3)
    → Dense(num_classes, softmax)
    """
    inp = keras.Input(shape=(sequence_len, num_features))
    x = layers.Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs=inp, outputs=out, name="cnn_v2")


cnn_model = build_cnn(SEQUENCE_LEN, NUM_FEATURES, NUM_CLASSES)
cnn_model.summary()

Model: "cnn_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 60, 126)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 60, 64)         │        24,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 30, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 15, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 15, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 204)            │        26,316 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 92,556 (361.55 KB)

 Trainable params: 92,172 (360.05 KB)

 Non-trainable params: 384 (1.50 KB)

In [7]:
cnn_checkpoint = str(MODEL_DIR / "cnn_hand_v2_best.keras")

cnn_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True),
        callbacks.ModelCheckpoint(cnn_checkpoint, monitor="val_accuracy", save_best_only=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6),
    ],
    verbose=1,
)

cnn_model = keras.models.load_model(cnn_checkpoint)
print(f"CNN model saved: {cnn_checkpoint}")

Epoch 1/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.1701 - loss: 3.9596 - val_accuracy: 0.3198 - val_loss: 3.1951 - learning_rate: 0.0010
Epoch 2/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3319 - loss: 2.5822 - val_accuracy: 0.4211 - val_loss: 2.4845 - learning_rate: 0.0010
Epoch 3/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4408 - loss: 1.9719 - val_accuracy: 0.4656 - val_loss: 2.3391 - learning_rate: 0.0010
Epoch 4/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5201 - loss: 1.6270 - val_accuracy: 0.4818 - val_loss: 2.2099 - learning_rate: 0.0010
Epoch 5/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5845 - loss: 1.3687 - val_accuracy: 0.5223 - val_loss: 2.3215 - learning_rate: 0.0010
Epoch 6/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6343 - loss: 1.1797 - val_accuracy: 0.5223 - val_loss: 2.2576 - learning_rate: 0.0010
Epoch 7/100
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6704 - loss: 1.

In [8]:
bigru_checkpoint = str(MODEL_DIR / "bigru_attn_v2_best.keras")
bigru_model = keras.models.load_model(bigru_checkpoint)
print(f"BiGRU+Attention model loaded from: {bigru_checkpoint}")

BiGRU+Attention model loaded from: ../saved_models/v2/bigru_attn_v2_best.keras


In [9]:
print("\nIndividual Model Results on Test Set:")
print("─" * 65)

bigru_scores = eval_model(bigru_model, X_test, y_test, "BiGRU + Attention")
gru_scores   = eval_model(gru_model,   X_test, y_test, "GRU")
cnn_scores   = eval_model(cnn_model,   X_test, y_test, "CNN")


Individual Model Results on Test Set:
─────────────────────────────────────────────────────────────────
  BiGRU + Attention          Top-1: 63.31%   Top-3: 79.84%   Top-5: 84.68%
  GRU                        Top-1: 54.64%   Top-3: 73.99%   Top-5: 82.66%
  CNN                        Top-1: 64.72%   Top-3: 80.85%   Top-5: 85.69%


In [10]:
# Predict probabilities from all 3 models
p_bigru = bigru_model.predict(X_test, verbose=0)
p_gru   = gru_model.predict(X_test,   verbose=0)
p_cnn   = cnn_model.predict(X_test,   verbose=0)

# Equal-weight average
p_ensemble = (p_bigru + p_gru + p_cnn) / 3.0

# Evaluate ensemble
true_labels = np.argmax(y_test, axis=1)

def topk_from_probs(probs, true_labels, k):
    topk_preds = np.argsort(probs, axis=1)[:, -k:]
    correct = sum(t in p for t, p in zip(true_labels, topk_preds))
    return correct / len(true_labels)

ens_top1 = topk_from_probs(p_ensemble, true_labels, k=1)
ens_top3 = topk_from_probs(p_ensemble, true_labels, k=3)
ens_top5 = topk_from_probs(p_ensemble, true_labels, k=5)

print(f"\n  {'Ensemble (avg 3 models)':<25}  Top-1: {ens_top1*100:.2f}%   Top-3: {ens_top3*100:.2f}%   Top-5: {ens_top5*100:.2f}%")


  Ensemble (avg 3 models)    Top-1: 66.94%   Top-3: 81.85%   Top-5: 86.90%


In [11]:
print("\n")
print(f"{'Model':<28} {'Top-1':>8} {'Top-3':>8} {'Top-5':>8}")
print("─" * 58)
print(f"{'BiGRU + Attention':<28} {bigru_scores[0]*100:>7.2f}% {bigru_scores[1]*100:>7.2f}% {bigru_scores[2]*100:>7.2f}%")
print(f"{'GRU':<28} {gru_scores[0]*100:>7.2f}% {gru_scores[1]*100:>7.2f}% {gru_scores[2]*100:>7.2f}%")
print(f"{'CNN':<28} {cnn_scores[0]*100:>7.2f}% {cnn_scores[1]*100:>7.2f}% {cnn_scores[2]*100:>7.2f}%")
print(f"{'Ensemble':<28} {ens_top1*100:>7.2f}% {ens_top3*100:>7.2f}% {ens_top5*100:>7.2f}%")
print("─" * 58)



Model                           Top-1    Top-3    Top-5
──────────────────────────────────────────────────────────
BiGRU + Attention              63.31%   79.84%   84.68%
GRU                            54.64%   73.99%   82.66%
CNN                            64.72%   80.85%   85.69%
Ensemble                       66.94%   81.85%   86.90%
──────────────────────────────────────────────────────────


In [12]:
# Find best single model by test Top-1
model_results = {
    "bigru": (bigru_scores[0], bigru_checkpoint),
    "gru":   (gru_scores[0],   gru_checkpoint),
    "cnn":   (cnn_scores[0],   cnn_checkpoint),
}

best_name, (best_top1, best_path) = max(model_results.items(), key=lambda x: x[1][0])
candidate_path = MODEL_DIR / "mudralearn_v2_candidate.keras"

shutil.copy(best_path, str(candidate_path))
print(f"Best single model : {best_name} — {best_top1*100:.2f}% Top-1")
print(f"Promoted to       : {candidate_path}")
print()
print("NOTE: Do NOT overwrite mudralearn_model.keras yet.")
print("      That happens only after notebook 08 (evaluation gate) PASSES.")

Best single model : cnn — 64.72% Top-1
Promoted to       : ../saved_models/v2/mudralearn_v2_candidate.keras

NOTE: Do NOT overwrite mudralearn_model.keras yet.
      That happens only after notebook 08 (evaluation gate) PASSES.


In [13]:
print("=" * 50)
print("FINDINGS SUMMARY")
print("=" * 50)
print(f"Best single model : {best_name} at {best_top1*100:.2f}% Top-1")
print(f"Ensemble Top-1    : {ens_top1*100:.2f}%")
print(f"Ensemble Top-3    : {ens_top3*100:.2f}%")
print(f"Ensemble Top-5    : {ens_top5*100:.2f}%")
print(f"Candidate saved   : {candidate_path}")
print("=" * 50)

FINDINGS SUMMARY
Best single model : cnn at 64.72% Top-1
Ensemble Top-1    : 66.94%
Ensemble Top-3    : 81.85%
Ensemble Top-5    : 86.90%
Candidate saved   : ../saved_models/v2/mudralearn_v2_candidate.keras
